<a href="https://colab.research.google.com/github/Obananob/Data-Science-Nigeria-2026-Hackathon-/blob/main/Dsn26.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import numpy as np
import pandas as pd
from sklearn.model_selection import KFold
from sklearn.metrics import mean_squared_error
from sklearn.preprocessing import LabelEncoder
import lightgbm as lgb
import xgboost as xgb
!pip install catboost
from catboost import CatBoostRegressor

pd.set_option('display.max_columns', None)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 97.1/97.1 MB 8.9 MB/s eta 0:00:00


In [ ]:
import pandas as pd
import numpy as np

train = pd.read_csv('/content/drive/MyDrive/DSN/train.csv')
test = pd.read_csv('/content/drive/MyDrive/DSN/test.csv')

print("Train shape:", train.shape, "| Test shape:", test.shape)
print(train.head())
print(train.isnull().sum())

TARGET = 'total_sales'
ID_COL = 'id'

train['is_train'] = 1
test['is_train'] = 0
test[TARGET] = np.nan

full = pd.concat([train, test], axis=0, ignore_index=True, sort=False)

Train shape: (6818, 13) | Test shape: (1705, 12)
          id product_code  product_weight_kg fat_content  shelf_visibility  \
0  row_00000   PRD-PRFP9S             14.252     Low Fat            0.0271   
1  row_00001   PRD-PXXK71              7.698     Low Fat            0.0720   
2  row_00002   PRD-V5MOIJ             14.264     Regular            0.0421   
3  row_00003   PRD-UN5Z3J                NaN     Regular            0.0449   
4  row_00004   PRD-6RDQYB             10.338     Regular            0.0120   

     product_category  product_price store_code  store_age_years store_size  \
0        Frozen Foods          81.37  STORE-AGY               45      Large   
1  HEALTH AND HYGIENE          42.05  STORE-YLW               35      Small   
2              Canned          41.35  STORE-89Z               33     Medium   
3         soft drinks         174.35  STORE-7WS               47     Medium   
4                meat         203.06  STORE-9RG               28      Small   

  store

In [ ]:

full['fat_content'] = full['fat_content'].astype(str).str.strip().str.lower()
full['fat_content'] = full['fat_content'].replace({
    'low fat': 'Low Fat', 'lowfat': 'Low Fat', 'low_fat': 'Low Fat', 'lf': 'Low Fat',
    'regular': 'Regular', 'reg': 'Regular'
})
full['fat_content'] = full['fat_content'].str.title()

full['product_category'] = full['product_category'].astype(str).str.strip().str.lower()
full['product_category'] = full['product_category'].str.title()

full['store_size'] = full['store_size'].astype(str).str.strip()
full['store_size'] = full['store_size'].replace({'nan': np.nan, 'Missing': np.nan})

full['store_location_tier'] = full['store_location_tier'].astype(str).str.strip()
full['store_format'] = full['store_format'].astype(str).str.strip()

In [ ]:
# product_weight_kg: impute per product_code first, then global median
weight_map = full.groupby('product_code')['product_weight_kg'].median()
full['product_weight_kg'] = full['product_weight_kg'].fillna(full['product_code'].map(weight_map))
full['product_weight_kg'] = full['product_weight_kg'].fillna(full['product_weight_kg'].median())

# store_size: impute using mode store_size for that store_format
size_mode_map = full.groupby('store_format')['store_size'].agg(
    lambda x: x.mode().iloc[0] if not x.mode().empty else 'Medium'
)
full['store_size'] = full['store_size'].fillna(full['store_format'].map(size_mode_map))
full['store_size'] = full['store_size'].fillna('Medium')

# shelf_visibility: 0 is a data artifact, treat as missing
full.loc[full['shelf_visibility'] == 0, 'shelf_visibility'] = np.nan
vis_map = full.groupby('product_category')['shelf_visibility'].transform('mean')
full['shelf_visibility'] = full['shelf_visibility'].fillna(vis_map)

print("Remaining missing values:\n", full.isnull().sum())

Remaining missing values:
 id                        0
product_code              0
product_weight_kg         0
fat_content               0
shelf_visibility          0
product_category          0
product_price             0
store_code                0
store_age_years           0
store_size                0
store_location_tier       0
store_format              0
total_sales            1705
is_train                  0
dtype: int64


In [ ]:
full['product_code_prefix'] = full['product_code'].astype(str).str[:2]

full['price_per_kg'] = full['product_price'] / full['product_weight_kg'].replace(0, np.nan)
full['price_per_kg'] = full['price_per_kg'].fillna(full['price_per_kg'].median())

full['price_bin'] = pd.qcut(full['product_price'], q=4, labels=False, duplicates='drop')

full['store_age_bucket'] = pd.cut(full['store_age_years'], bins=[-1, 5, 10, 15, 100], labels=False)

train_mask = full['is_train'] == 1
global_mean = full.loc[train_mask, TARGET].mean()

for col in ['product_code', 'store_code', 'product_category', 'store_format',
            'store_location_tier', 'store_size', 'product_code_prefix']:
    agg = full.loc[train_mask].groupby(col)[TARGET].mean()
    full[f'{col}_avg_sales'] = full[col].map(agg).fillna(global_mean)

store_price_avg = full.loc[train_mask].groupby('store_code')['product_price'].mean()
full['store_avg_price'] = full['store_code'].map(store_price_avg)
full['store_avg_price'] = full['store_avg_price'].fillna(full['product_price'].mean())

full['format_tier'] = full['store_format'] + '_' + full['store_location_tier']

In [ ]:
cat_cols = ['fat_content', 'product_category', 'store_size', 'store_location_tier',
            'store_format', 'product_code_prefix', 'format_tier']

for col in cat_cols:
    le = LabelEncoder()
    full[col] = le.fit_transform(full[col].astype(str))

full = full.drop(columns=['product_code', 'store_code'])

In [ ]:
train_final = full[full['is_train'] == 1].drop(columns=['is_train'])
test_final = full[full['is_train'] == 0].drop(columns=['is_train'])

X = train_final.drop(columns=[TARGET, ID_COL])
y = train_final[TARGET]
X_test = test_final.drop(columns=[TARGET, ID_COL])
test_ids = test_final[ID_COL]

print("Final features:", X.columns.tolist())
print("X shape:", X.shape, "X_test shape:", X_test.shape)

Final features: ['product_weight_kg', 'fat_content', 'shelf_visibility', 'product_category', 'product_price', 'store_age_years', 'store_size', 'store_location_tier', 'store_format', 'product_code_prefix', 'price_per_kg', 'price_bin', 'store_age_bucket', 'product_code_avg_sales', 'store_code_avg_sales', 'product_category_avg_sales', 'store_format_avg_sales', 'store_location_tier_avg_sales', 'store_size_avg_sales', 'product_code_prefix_avg_sales', 'store_avg_price', 'format_tier']
X shape: (6818, 22) X_test shape: (1705, 22)


In [ ]:
N_FOLDS = 5
kf = KFold(n_splits=N_FOLDS, shuffle=True, random_state=42)

oof_lgb = np.zeros(len(X)); oof_xgb = np.zeros(len(X)); oof_cat = np.zeros(len(X))
pred_lgb = np.zeros(len(X_test)); pred_xgb = np.zeros(len(X_test)); pred_cat = np.zeros(len(X_test))

lgb_params = dict(objective='regression', metric='rmse', learning_rate=0.03,
                   num_leaves=31, feature_fraction=0.8, bagging_fraction=0.8,
                   bagging_freq=5, verbose=-1)

xgb_params = dict(objective='reg:squarederror', eval_metric='rmse',
                   learning_rate=0.03, max_depth=6, subsample=0.8, colsample_bytree=0.8)

for fold, (tr_idx, val_idx) in enumerate(kf.split(X)):
    print(f"\n===== Fold {fold+1}/{N_FOLDS} =====")
    X_tr, X_val = X.iloc[tr_idx], X.iloc[val_idx]
    y_tr, y_val = y.iloc[tr_idx], y.iloc[val_idx]

    lgb_train = lgb.Dataset(X_tr, y_tr)
    lgb_val = lgb.Dataset(X_val, y_val, reference=lgb_train)
    model_lgb = lgb.train(lgb_params, lgb_train, num_boost_round=3000,
                           valid_sets=[lgb_val],
                           callbacks=[lgb.early_stopping(100), lgb.log_evaluation(0)])
    oof_lgb[val_idx] = model_lgb.predict(X_val)
    pred_lgb += model_lgb.predict(X_test) / N_FOLDS

    model_xgb = xgb.XGBRegressor(**xgb_params, n_estimators=3000)
    model_xgb.fit(X_tr, y_tr, eval_set=[(X_val, y_val)], verbose=False)
    oof_xgb[val_idx] = model_xgb.predict(X_val)
    pred_xgb += model_xgb.predict(X_test) / N_FOLDS

    model_cat = CatBoostRegressor(iterations=3000, learning_rate=0.03, depth=6,
                                   loss_function='RMSE', verbose=False, random_state=42)
    model_cat.fit(X_tr, y_tr, eval_set=(X_val, y_val), early_stopping_rounds=100)
    oof_cat[val_idx] = model_cat.predict(X_val)
    pred_cat += model_cat.predict(X_test) / N_FOLDS

    print(f"Fold {fold+1} LGB RMSE: {np.sqrt(mean_squared_error(y_val, oof_lgb[val_idx])):.4f}")


===== Fold 1/5 =====
Training until validation scores don't improve for 100 rounds
Early stopping, best iteration is:
[95]	valid_0's rmse: 984.703
Fold 1 LGB RMSE: 984.7032

===== Fold 2/5 =====
Training until validation scores don't improve for 100 rounds
Early stopping, best iteration is:
[102]	valid_0's rmse: 944.246
Fold 2 LGB RMSE: 944.2461

===== Fold 3/5 =====
Training until validation scores don't improve for 100 rounds
Early stopping, best iteration is:
[85]	valid_0's rmse: 1007.96
Fold 3 LGB RMSE: 1007.9574

===== Fold 4/5 =====
Training until validation scores don't improve for 100 rounds
Early stopping, best iteration is:
[99]	valid_0's rmse: 1014.78
Fold 4 LGB RMSE: 1014.7775

===== Fold 5/5 =====
Training until validation scores don't improve for 100 rounds
Early stopping, best iteration is:
[90]	valid_0's rmse: 1008.55
Fold 5 LGB RMSE: 1008.5500


In [ ]:
rmse_lgb = np.sqrt(mean_squared_error(y, oof_lgb))
rmse_xgb = np.sqrt(mean_squared_error(y, oof_xgb))
rmse_cat = np.sqrt(mean_squared_error(y, oof_cat))
print(f"OOF RMSE — LGB: {rmse_lgb:.4f} | XGB: {rmse_xgb:.4f} | CAT: {rmse_cat:.4f}")

best_rmse = np.sqrt(mean_squared_error(y, (oof_lgb+oof_xgb+oof_cat)/3))
best_weights = (1/3, 1/3, 1/3)
for w1 in np.arange(0, 1.05, 0.1):
    for w2 in np.arange(0, 1.05 - w1, 0.1):
        w3 = 1 - w1 - w2
        if w3 < 0: continue
        r = np.sqrt(mean_squared_error(y, w1*oof_lgb + w2*oof_xgb + w3*oof_cat))
        if r < best_rmse:
            best_rmse, best_weights = r, (w1, w2, w3)

print(f"Best blend RMSE: {best_rmse:.4f} with weights {best_weights}")

OOF RMSE — LGB: 992.3819 | XGB: 1135.1378 | CAT: 966.1842
Best blend RMSE: 966.1842 with weights (np.float64(0.0), np.float64(0.0), np.float64(1.0))


In [ ]:

w1, w2, w3 = best_weights
final_pred = np.clip(w1*pred_lgb + w2*pred_xgb + w3*pred_cat, 0, None)

submission = pd.DataFrame({ID_COL: test_ids, TARGET: final_pred})
submission.to_csv('submission.csv', index=False)
print("Saved submission.csv")
print(submission.head())

Saved submission.csv
             id  total_sales
6818  row_00009  2709.825312
6819  row_00015  5531.672543
6820  row_00019  2826.563164
6821  row_00020  3474.799426
6822  row_00023  2260.499908


In [ ]:

from sklearn.model_selection import KFold

y_log = np.log1p(y)  # log1p handles it safely even if any value is 0

N_FOLDS = 5
kf = KFold(n_splits=N_FOLDS, shuffle=True, random_state=42)

oof_cat_log = np.zeros(len(X))
pred_cat_log = np.zeros(len(X_test))

cat_params = dict(
    iterations=5000,
    learning_rate=0.02,
    depth=7,
    l2_leaf_reg=5,
    loss_function='RMSE',
    verbose=False,
    random_state=42
)

for fold, (tr_idx, val_idx) in enumerate(kf.split(X)):
    X_tr, X_val = X.iloc[tr_idx], X.iloc[val_idx]
    y_tr, y_val = y_log.iloc[tr_idx], y_log.iloc[val_idx]

    model = CatBoostRegressor(**cat_params)
    model.fit(X_tr, y_tr, eval_set=(X_val, y_val), early_stopping_rounds=150)

    oof_cat_log[val_idx] = model.predict(X_val)
    pred_cat_log += model.predict(X_test) / N_FOLDS

# Convert back from log space to real sales values
oof_cat_log_real = np.expm1(oof_cat_log)
rmse_log = np.sqrt(mean_squared_error(y, oof_cat_log_real))
print(f"CatBoost (log-target) OOF RMSE: {rmse_log:.4f}")
print(f"Previous best (no log): 966.1842")

CatBoost (log-target) OOF RMSE: 998.2563
Previous best (no log): 966.1842


In [ ]:
if rmse_log < 966.1842:
    print("Log-transform wins. Using it for final submission.")
    final_pred = np.clip(np.expm1(pred_cat_log), 0, None)
else:
    print("Original scale wins. Keeping prior CatBoost predictions.")
    final_pred = np.clip(pred_cat, 0, None)

submission = pd.DataFrame({ID_COL: test_ids, TARGET: final_pred})
submission.to_csv('submission.csv', index=False)
print("Saved submission.csv")
print(submission.head())

In [ ]:
importances = model_cat.get_feature_importance()
feat_imp = pd.DataFrame({'feature': X.columns, 'importance': importances})
feat_imp = feat_imp.sort_values('importance', ascending=False)
print(feat_imp.to_string(index=False))

                      feature  importance
       product_code_avg_sales   36.679331
                  format_tier   16.534933
       store_format_avg_sales    9.772580
         store_code_avg_sales    8.509035
              store_avg_price    7.513135
                 store_format    7.497165
                product_price    4.998888
              store_age_years    2.825923
                    price_bin    2.204559
         store_size_avg_sales    0.611984
             shelf_visibility    0.587355
            product_weight_kg    0.517647
                 price_per_kg    0.507053
   product_category_avg_sales    0.340525
             product_category    0.252502
          store_location_tier    0.218026
                   store_size    0.169167
store_location_tier_avg_sales    0.144457
                  fat_content    0.115735
          product_code_prefix    0.000000
             store_age_bucket    0.000000
product_code_prefix_avg_sales    0.000000


In [ ]:
import itertools
from sklearn.model_selection import KFold

param_grid = {
    'depth': [5, 6, 7, 8],
    'learning_rate': [0.01, 0.02, 0.03, 0.05],
    'l2_leaf_reg': [1, 3, 5, 7, 9]
}


np.random.seed(42)
combos = list(itertools.product(param_grid['depth'], param_grid['learning_rate'], param_grid['l2_leaf_reg']))
sampled = [combos[i] for i in np.random.choice(len(combos), size=15, replace=False)]

kf = KFold(n_splits=5, shuffle=True, random_state=42)
results = []

for depth, lr, l2 in sampled:
    oof = np.zeros(len(X))
    for tr_idx, val_idx in kf.split(X):
        X_tr, X_val = X.iloc[tr_idx], X.iloc[val_idx]
        y_tr, y_val = y.iloc[tr_idx], y.iloc[val_idx]
        m = CatBoostRegressor(iterations=3000, depth=depth, learning_rate=lr,
                               l2_leaf_reg=l2, loss_function='RMSE',
                               verbose=False, random_state=42)
        m.fit(X_tr, y_tr, eval_set=(X_val, y_val), early_stopping_rounds=100)
        oof[val_idx] = m.predict(X_val)
    rmse = np.sqrt(mean_squared_error(y, oof))
    results.append((depth, lr, l2, rmse))
    print(f"depth={depth}, lr={lr}, l2={l2} -> RMSE={rmse:.4f}")

results_df = pd.DataFrame(results, columns=['depth', 'lr', 'l2', 'rmse']).sort_values('rmse')
print("\nBest params:\n", results_df.head())

depth=6, lr=0.03, l2=1 -> RMSE=965.6304
depth=5, lr=0.01, l2=1 -> RMSE=964.3503
depth=6, lr=0.01, l2=5 -> RMSE=965.6995
depth=6, lr=0.03, l2=3 -> RMSE=966.1842
depth=5, lr=0.05, l2=7 -> RMSE=964.5529
depth=6, lr=0.02, l2=7 -> RMSE=966.1138
depth=5, lr=0.03, l2=1 -> RMSE=965.7226
depth=8, lr=0.03, l2=1 -> RMSE=969.2654
depth=5, lr=0.01, l2=9 -> RMSE=964.7005
depth=5, lr=0.03, l2=5 -> RMSE=964.8636
depth=7, lr=0.02, l2=9 -> RMSE=967.5547
depth=6, lr=0.03, l2=7 -> RMSE=965.5773
depth=8, lr=0.02, l2=5 -> RMSE=968.5981
depth=6, lr=0.05, l2=1 -> RMSE=965.1083
depth=8, lr=0.02, l2=7 -> RMSE=968.2017

Best params:
     depth    lr  l2        rmse
1       5  0.01   1  964.350280
4       5  0.05   7  964.552921
8       5  0.01   9  964.700535
9       5  0.03   5  964.863569
13      6  0.05   1  965.108255


In [ ]:
drop_zero_imp = ['product_code_prefix', 'store_age_bucket', 'product_code_prefix_avg_sales']

X_pruned = X.drop(columns=drop_zero_imp)
X_test_pruned = X_test.drop(columns=drop_zero_imp)

oof_cat_pruned = np.zeros(len(X_pruned))
pred_cat_pruned = np.zeros(len(X_test_pruned))

kf = KFold(n_splits=5, shuffle=True, random_state=42)

for tr_idx, val_idx in kf.split(X_pruned):
    X_tr, X_val = X_pruned.iloc[tr_idx], X_pruned.iloc[val_idx]
    y_tr, y_val = y.iloc[tr_idx], y.iloc[val_idx]

    m = CatBoostRegressor(iterations=3000, learning_rate=0.03, depth=6,
                           loss_function='RMSE', verbose=False, random_state=42)
    m.fit(X_tr, y_tr, eval_set=(X_val, y_val), early_stopping_rounds=100)
    oof_cat_pruned[val_idx] = m.predict(X_val)
    pred_cat_pruned += m.predict(X_test_pruned) / 5

rmse_pruned = np.sqrt(mean_squared_error(y, oof_cat_pruned))
print(f"Pruned CatBoost OOF RMSE: {rmse_pruned:.4f}")
print(f"Previous best: 966.1842")

Pruned CatBoost OOF RMSE: 966.1842
Previous best: 966.1842


In [ ]:
oof_final = np.zeros(len(X_pruned))
pred_final = np.zeros(len(X_test_pruned))

kf = KFold(n_splits=5, shuffle=True, random_state=42)

for tr_idx, val_idx in kf.split(X_pruned):
    X_tr, X_val = X_pruned.iloc[tr_idx], X_pruned.iloc[val_idx]
    y_tr, y_val = y.iloc[tr_idx], y.iloc[val_idx]

    m = CatBoostRegressor(iterations=5000, depth=5, learning_rate=0.01, l2_leaf_reg=1,
                           loss_function='RMSE', verbose=False, random_state=42)
    m.fit(X_tr, y_tr, eval_set=(X_val, y_val), early_stopping_rounds=150)
    oof_final[val_idx] = m.predict(X_val)
    pred_final += m.predict(X_test_pruned) / 5

rmse_final = np.sqrt(mean_squared_error(y, oof_final))
print(f"Final model OOF RMSE: {rmse_final:.4f}")

Final model OOF RMSE: 964.3503


In [ ]:
final_pred = np.clip(pred_final, 0, None)
submission = pd.DataFrame({ID_COL: test_ids, TARGET: final_pred})
submission.to_csv('submission1.csv', index=False)
print("Saved submission.csv")
print(submission.head())

Saved submission.csv
             id  total_sales
6818  row_00009  2717.356094
6819  row_00015  5613.916399
6820  row_00019  2777.959741
6821  row_00020  3535.916351
6822  row_00023  2301.404883


In [ ]:
from sklearn.model_selection import KFold

encode_cols = ['product_code', 'store_code', 'product_category', 'store_format',
               'store_location_tier', 'store_size', 'product_code_prefix']


X_raw = train_final.drop(columns=[TARGET, ID_COL])
X_test_raw = test_final.drop(columns=[TARGET, ID_COL])


leaked_cols = [c for c in X_raw.columns if c.endswith('_avg_sales')]
X_raw = X_raw.drop(columns=leaked_cols + ['product_code_prefix', 'store_age_bucket'], errors='ignore')
X_test_raw = X_test_raw.drop(columns=leaked_cols + ['product_code_prefix', 'store_age_bucket'], errors='ignore')

print("Columns going into fold-safe encoding:", X_raw.columns.tolist())

N_FOLDS = 5
kf = KFold(n_splits=N_FOLDS, shuffle=True, random_state=42)

oof_safe = np.zeros(len(X_raw))
pred_safe = np.zeros(len(X_test_raw))
global_mean = y.mean()

for fold, (tr_idx, val_idx) in enumerate(kf.split(X_raw)):
    X_tr, X_val = X_raw.iloc[tr_idx].copy(), X_raw.iloc[val_idx].copy()
    y_tr, y_val = y.iloc[tr_idx], y.iloc[val_idx]
    X_te = X_test_raw.copy()

    # Recompute target encodings using ONLY this fold's training rows
    tr_with_target = X_tr.copy()
    tr_with_target[TARGET] = y_tr.values

    for col in ['fat_content', 'product_category', 'store_format',
                'store_location_tier', 'store_size', 'format_tier']:
        # smoothing: blend group mean with global mean, weighted by count
        agg = tr_with_target.groupby(col)[TARGET].agg(['mean', 'count'])
        smoothing = 10  # higher = more shrinkage toward global mean for rare categories
        smoothed = (agg['mean'] * agg['count'] + global_mean * smoothing) / (agg['count'] + smoothing)

        X_tr[f'{col}_avg_sales'] = X_tr[col].map(smoothed).fillna(global_mean)
        X_val[f'{col}_avg_sales'] = X_val[col].map(smoothed).fillna(global_mean)
        X_te[f'{col}_avg_sales'] = X_te[col].map(smoothed).fillna(global_mean)

    m = CatBoostRegressor(iterations=5000, depth=5, learning_rate=0.01, l2_leaf_reg=1,
                           loss_function='RMSE', verbose=False, random_state=42)
    m.fit(X_tr, y_tr, eval_set=(X_val, y_val), early_stopping_rounds=150)

    oof_safe[val_idx] = m.predict(X_val)
    pred_safe += m.predict(X_te) / N_FOLDS

    print(f"Fold {fold+1} done")

rmse_safe = np.sqrt(mean_squared_error(y, oof_safe))
print(f"\nFold-safe OOF RMSE: {rmse_safe:.4f}")

Columns going into fold-safe encoding: ['product_weight_kg', 'fat_content', 'shelf_visibility', 'product_category', 'product_price', 'store_age_years', 'store_size', 'store_location_tier', 'store_format', 'price_per_kg', 'price_bin', 'store_avg_price', 'format_tier']
Fold 1 done
Fold 2 done
Fold 3 done
Fold 4 done
Fold 5 done

Fold-safe OOF RMSE: 1078.5147
This number should now actually match your leaderboard score, not diverge from it.


In [ ]:
final_pred_safe = np.clip(pred_safe, 0, None)
submission_safe = pd.DataFrame({ID_COL: test_ids, TARGET: final_pred_safe})
submission_safe.to_csv('submission_v2.csv', index=False)
print("Saved submission_v2.csv")
print(submission_safe.head())

Saved submission_v2.csv
             id  total_sales
6818  row_00009  2866.696665
6819  row_00015  4137.660167
6820  row_00019  2978.940578
6821  row_00020  3332.878688
6822  row_00023  2544.656936
